In [ ]:
using CSV
using DataFrames
using Flux
using Dates
using Statistics
using StatsBase
using Random

# 1. INCLUDIAMO LE UTILS
include("utils.jl") 

# ============================================================================
# 2. CARICAMENTO E PREPARAZIONE DATI
# ============================================================================
println("--- FASE 1: Caricamento e Pulizia Dati ---")

filename = "Fraudulent_E-Commerce_Transaction_Data_merge.csv"
df_original = CSV.read(filename, DataFrame)

# --- Feature Engineering ---
# 1. Logaritmo su importo
df_original."LogAmount" = log.(df_original."Transaction Amount" .+ 1)

# 2. IP Frequency
ip_counts = countmap(df_original."IP Address")
df_original.IP_Frequency = [ip_counts[ip] for ip in df_original."IP Address"]

# 3. Transaction Hour
df_original."Transaction Hour" = Float32.(df_original."Transaction Hour")

# --- SELEZIONE COLONNE ---
input_cols = ["LogAmount", "Transaction Hour", "IP_Frequency", "Account Age Days"]
target_col = "Is Fraudulent"
id_col = "Transaction ID"

# ============================================================================
# 3. STANDARDIZZAZIONE Z-SCORE (LA SOLUZIONE AL PROBLEMA "MEDIO")
# ============================================================================
println("Applicazione Z-Score (Forziamo i dati in una scala leggibile)...")

# Funzione per standardizzare: (Valore - Media) / DeviazioneStandard
# Questo mette tutti i dati sulla stessa "lunghezza d'onda"
function z_score_normalize!(df, cols)
    stats = Dict()
    for col in cols
        data = df[:, col]
        μ = mean(data) # Media
        σ = std(data)  # Deviazione Standard
        
        # Salviamo i parametri per usarli dopo
        stats[col] = (μ, σ)
        
        # Applichiamo la trasformazione
        df[!, col] = (df[:, col] .- μ) ./ (σ + 1e-8)
    end
    return stats
end

# Standardizziamo TUTTO il dataset subito
normalization_stats = z_score_normalize!(df_original, input_cols)

println("Dati standardizzati. Esempio IP Frequency prima riga: $(df_original[1, "IP_Frequency"])")

# ============================================================================
# 4. BILANCIAMENTO DATASET
# ============================================================================
fraud_rows = df_original[df_original[:, target_col] .== 1, :]
legit_rows = df_original[df_original[:, target_col] .== 0, :]

n_min = min(nrow(fraud_rows), nrow(legit_rows))

# Creiamo dataset bilanciato 50/50
df_train = vcat(
    fraud_rows[randperm(nrow(fraud_rows))[1:n_min], :],
    legit_rows[randperm(nrow(legit_rows))[1:n_min], :]
)
df_train = df_train[randperm(nrow(df_train)), :]

println("Dataset Bilanciato: $(nrow(df_train)) righe.")

train_inputs = Matrix{Float32}(df_train[:, input_cols])
train_targets = df_train[:, target_col] .== 1 

# ============================================================================
# 5. TRAINING
# ============================================================================
println("\n--- FASE 2: Training ---")

# Aumentiamo i neuroni per gestire la complessità
topology = [32, 32, 16] 
learning_rate = 0.01
max_epochs = 500
k_folds = 5 

# Usiamo ReLU per evitare che i neuroni muoiano (smettano di imparare)
functions = [Flux.relu, Flux.relu, Flux.relu]

cv_indices = crossvalidation(train_targets, k_folds)

results = ANNCrossValidation(
    topology, 
    (train_inputs, train_targets), 
    cv_indices;
    maxEpochs=max_epochs, 
    learningRate=learning_rate,
    numExecutions=1,
    transferFunctions=functions
)

(meanAcc, stdAcc), (meanErr, stdErr), (meanSens, stdSens), 
(meanSpec, stdSpec), (meanPPV, stdPPV), _, (meanF1, stdF1), confMatrix = results

println("\n=== REPORT PRESTAZIONI ===")
println("Accuratezza: $(round(meanAcc * 100, digits=2))%")
println("Sensibilità: $(round(meanSens * 100, digits=2))%")
println("Specificità: $(round(meanSpec * 100, digits=2))%")

println("\nMatrice di Confusione:")
display(confMatrix)

# ============================================================================
# 6. EXPORT FINALE
# ============================================================================
println("\n--- FASE 3: Export Finale ---")

# Nota: i dati sono GIA' standardizzati, quindi passiamo train_inputs diretti
train_targets_matrix = reshape(train_targets, :, 1)

println("Addestramento modello finale...")
final_model, _ = trainClassANN(topology, (train_inputs, train_targets_matrix); 
                               maxEpochs=max_epochs, 
                               learningRate=learning_rate,
                               transferFunctions=functions)

# Prendiamo tutti i dati (sono già stati standardizzati all'inizio!)
all_inputs = Matrix{Float32}(df_original[:, input_cols])

println("Calcolo probabilità...")
probabilities = final_model(all_inputs')' 

# --- NUOVA LOGICA SOGLIE ---
# Dato che il modello ora è più aggressivo, usiamo soglie più pulite
function get_risk_label(p)
    if p < 0.3 return "Basso"
    elseif p < 0.7 return "Medio"
    else return "ALTO"
    end
end

risk_levels = get_risk_label.(probabilities)

output_df = DataFrame(
    Transaction_ID = df_original[:, id_col],
    Fraud_Probability = round.(vec(probabilities), digits=4), 
    Risk_Level = vec(risk_levels)
)

output_filename = "report_transazioni_rischio.csv"
CSV.write(output_filename, output_df)

println("✅ Analisi completata. Controlla '$output_filename'")

--- FASE 1: Caricamento e Pulizia Dati ---
Applicazione Z-Score (Forziamo i dati in una scala leggibile)...
Dati standardizzati. Esempio IP Frequency prima riga: -0.02029217292972786
Dataset Bilanciato: 150120 righe.

--- FASE 2: Training ---

Fold 1/5
  Execution 1/1

Fold 2/5
  Execution 1/1

Fold 3/5
  Execution 1/1

Fold 4/5
  Execution 1/1

Fold 5/5
  Execution 1/1

=== REPORT PRESTAZIONI ===
Accuratezza: 74.45%
Sensibilità: 77.75%
Specificità: 71.16%

Matrice di Confusione:


2×2 Matrix{Float64}:
 53412.0  21648.0
 16702.0  58358.0